# THP vs RoTHP vs HoTHP — Dataset Sintético

Treina e avalia os 3 modelos no dataset `exp_short_memory` (memória curta exponencial, 4 tipos de evento, ~300 sequências).

**Runtime → Change runtime type → GPU (A100 ou T4)**

In [ ]:
# ── 1. SETUP ──────────────────────────────────────────────────────────
# Clona o repositório e instala as dependências.

import os, sys

if not os.path.exists('ufc-easytpp'):
    os.system('git clone https://github.com/hugoramos/ufc-easytpp.git')

sys.path.insert(0, 'ufc-easytpp')

os.system('pip install omegaconf datasets pyyaml matplotlib -q')

# O Runner encontra modelos via __subclasses__() — só enxerga classes que
# foram importadas. Este bloco garante que THP, RoTHP e HoTHP são importados.
with open('ufc-easytpp/easy_tpp/model/__init__.py', 'w') as f:
    f.write(
        'from easy_tpp.model.torch_model.torch_basemodel import TorchBaseModel\n'
        'from easy_tpp.model.torch_model.torch_thp    import THP\n'
        'from easy_tpp.model.torch_model.torch_rothp  import RoTHP\n'
        'from easy_tpp.model.torch_model.torch_hothp  import HoTHP\n'
    )

import torch
GPU = 0 if torch.cuda.is_available() else -1
print(f'GPU={GPU}  torch={torch.__version__}')

In [ ]:
# ── 2. PARÂMETROS ─────────────────────────────────────────────────────
# Todos os parâmetros do experimento ficam aqui.

import gc
import tempfile
import yaml
import numpy as np
import matplotlib.pyplot as plt

import easy_tpp.model   # garante que as subclasses ficam registradas
from easy_tpp.config_factory import Config
from easy_tpp.runner import Runner

MODELS      = ['THP', 'RoTHP', 'HoTHP']
SEED        = 2019
MAX_EPOCH   = 30       # épocas de treino
BATCH_SIZE  = 64
HIDDEN_SIZE = 32
NUM_HEADS   = 2
NUM_LAYERS  = 2
LR          = 1e-3

# Dataset sintético incluso no repositório
DATASET_DIR = 'ufc-easytpp/datasets/synthetic_hothp_scenarios/exp_short_memory'
NUM_TYPES   = 4    # número de tipos de evento
PAD_ID      = 4    # id do token de padding (= num_types)
MAX_LEN     = 70   # comprimento máximo das sequências

print('Parâmetros configurados.')

In [ ]:
# ── 3. TREINAMENTO ────────────────────────────────────────────────────
# Para cada modelo: monta o config, treina, salva o caminho do checkpoint.

checkpoint_dirs = {}   # model_id → caminho do checkpoint treinado

for model_id in MODELS:
    print(f'\n=== Treinando {model_id} ===')

    # Config completo como dicionário Python — sem arquivos externos
    config = {
        'pipeline_config_id': 'runner_config',
        'data': {
            'synthetic': {
                'data_format': 'json',
                'train_dir':  DATASET_DIR,
                'valid_dir':  DATASET_DIR,
                'test_dir':   DATASET_DIR,
                'data_specs': {
                    'num_event_types': NUM_TYPES,
                    'pad_token_id':    PAD_ID,
                    'padding_side':    'right',
                    'truncation_side': 'right',
                    'max_len':         MAX_LEN,
                },
            },
        },
        f'{model_id}_train': {
            'base_config': {
                'stage':      'train',
                'backend':    'torch',
                'dataset_id': 'synthetic',
                'runner_id':  'std_tpp',
                'model_id':   model_id,
                'base_dir':   f'./checkpoints/{model_id}/',
            },
            'trainer_config': {
                'batch_size':    BATCH_SIZE,
                'max_epoch':     MAX_EPOCH,
                'seed':          SEED,
                'gpu':           GPU,
                'metrics':       ['acc', 'rmse'],
                'valid_freq':    5,
                'use_tfb':       False,
                'optimizer':     'adam',
                'learning_rate': LR,
                'shuffle':       False,
            },
            'model_config': {
                'hidden_size':  HIDDEN_SIZE,
                'num_heads':    NUM_HEADS,
                'num_layers':   NUM_LAYERS,
                'dropout':      0.1,
                'time_emb_size': 16,
                'use_ln':       False,
                'loss_integral_num_sample_per_step': 20,
                'mc_num_sample_per_step': 20,
                'thinning': {
                    'num_sample': 1, 'num_exp': 500, 'look_ahead_time': 10,
                    'patience_counter': 5, 'over_sample_rate': 5,
                    'num_samples_boundary': 5, 'dtime_max': 10,
                    'num_seq': 10, 'num_step_gen': 1,
                },
            },
        },
    }

    # Escreve o config em arquivo temporário e carrega
    with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
        yaml.dump(config, f)
        tmp = f.name

    cfg    = Config.build_from_yaml_file(tmp, experiment_id=f'{model_id}_train')
    runner = Runner.build_from_config(cfg)
    runner.run()
    os.unlink(tmp)

    # Encontra o checkpoint salvo (EasyTPP cria um subdiretório com timestamp)
    base = f'./checkpoints/{model_id}'
    ckpt = sorted(
        [os.path.join(base, d, 'models', 'saved_model')
         for d in os.listdir(base)
         if os.path.exists(os.path.join(base, d, 'models', 'saved_model'))]
    )[-1]
    checkpoint_dirs[model_id] = ckpt
    print(f'  Checkpoint: {ckpt}')

    del runner
    gc.collect()
    torch.cuda.empty_cache()

print('\nTreinamento concluído!')

In [ ]:
# ── 4. AVALIAÇÃO ──────────────────────────────────────────────────────
# Carrega cada modelo treinado e avalia no conjunto de teste.

results = {}   # model_id → {'nll', 'acc', 'rmse'}

for model_id in MODELS:
    print(f'\n=== Avaliando {model_id} ===')

    config = {
        'pipeline_config_id': 'runner_config',
        'data': {
            'synthetic': {
                'data_format': 'json',
                'train_dir':  DATASET_DIR,
                'valid_dir':  DATASET_DIR,
                'test_dir':   DATASET_DIR,
                'data_specs': {
                    'num_event_types': NUM_TYPES,
                    'pad_token_id':    PAD_ID,
                    'padding_side':    'right',
                    'truncation_side': 'right',
                    'max_len':         MAX_LEN,
                },
            },
        },
        f'{model_id}_eval': {
            'base_config': {
                'stage':      'eval',
                'backend':    'torch',
                'dataset_id': 'synthetic',
                'runner_id':  'std_tpp',
                'model_id':   model_id,
                'base_dir':   f'./checkpoints/{model_id}/',
            },
            'trainer_config': {
                'batch_size': BATCH_SIZE,
                'max_epoch':  1,
                'seed':       SEED,
                'gpu':        GPU,
                'metrics':    ['acc', 'rmse'],
            },
            'model_config': {
                'hidden_size':       HIDDEN_SIZE,
                'num_heads':         NUM_HEADS,
                'num_layers':        NUM_LAYERS,
                'dropout':           0.1,
                'time_emb_size':     16,
                'use_ln':            False,
                'pretrained_model_dir': checkpoint_dirs[model_id],
                'thinning': {
                    'num_sample': 1, 'num_exp': 500, 'look_ahead_time': 10,
                    'patience_counter': 5, 'over_sample_rate': 5,
                    'num_samples_boundary': 5, 'dtime_max': 10,
                    'num_seq': 10, 'num_step_gen': 1,
                },
            },
        },
    }

    with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
        yaml.dump(config, f)
        tmp = f.name

    cfg    = Config.build_from_yaml_file(tmp, experiment_id=f'{model_id}_eval')
    runner = Runner.build_from_config(cfg)
    os.unlink(tmp)

    metrics = runner._evaluate_model(runner._data_loader.test_loader())
    results[model_id] = {
        'nll':  -metrics['loglike'],
        'acc':   metrics['acc'],
        'rmse':  metrics['rmse'],
    }
    print(f'  NLL={results[model_id]["nll"]:.4f}  '
          f'ACC={results[model_id]["acc"]:.4f}  '
          f'RMSE={results[model_id]["rmse"]:.4f}')

    del runner
    gc.collect()
    torch.cuda.empty_cache()

print('\nAvaliação concluída!')

In [ ]:
# ── 5. RESULTADOS ─────────────────────────────────────────────────────

# Tabela
print(f'{"Modelo":<10} {"NLL ↓":>10} {"ACC ↑":>10} {"RMSE ↓":>10}')
print('-' * 44)
for model_id in MODELS:
    r = results[model_id]
    print(f'{model_id:<10} {r["nll"]:>10.4f} {r["acc"]:>10.4f} {r["rmse"]:>10.4f}')

# Gráfico
COLORS = {'THP': '#4C72B0', 'RoTHP': '#55A868', 'HoTHP': '#C44E52'}
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, (metric, label, better) in zip(axes, [
    ('nll',  'NLL (↓ melhor)',      'min'),
    ('acc',  'Accuracy (↑ melhor)', 'max'),
    ('rmse', 'RMSE (↓ melhor)',     'min'),
]):
    values = [results[m][metric] for m in MODELS]
    colors = [COLORS[m] for m in MODELS]
    bars = ax.bar(MODELS, values, color=colors, alpha=0.85)

    # Destaca o melhor
    best_idx = values.index(min(values) if better == 'min' else max(values))
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(3)

    ax.set_title(label, fontweight='bold')
    ax.set_ylabel(metric.upper())
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('THP vs RoTHP vs HoTHP — exp_short_memory', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('resultado_sintetico.png', dpi=150, bbox_inches='tight')
plt.show()